In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [2]:
def prepare_dataset(input_csv):
    # defining the column names that are required - ignoring columns like id and source
    print("Loading dataset...")
    df = pd.read_csv(input_csv, header=0)
    print (df.info())
    
    df = df.drop(columns=['id', 'dataset'])  # Drop 'id' and 'source' if they exist
    df['chol'] = pd.to_numeric(df['chol'])
    df['age'] = pd.to_numeric(df['age'])

    df['target'] = df['num'].apply(lambda x: 1 if x > 0 else 0)  # Create binary target variable
    df = df.drop(columns=['num'])  # Drop original 'num' column as it's now represented in 'target'
    
    #numberical columns to be imputed with mean
    num_cols = df.select_dtypes(include=['float64', 'int64'])  # Exclude target variable
    # Categorical columns to be One-Hot Encoded
    cat_cols = df.select_dtypes(include=['object', 'category']).columns
    print(num_cols)
    print(cat_cols)
    print("--- Imputing Missing Values ---")

    # A. Mean Fill for Numerical Columns
    for col in num_cols:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].astype(float)
            mean_val = df[col].mean()
            df[col] = df[col].fillna(mean_val)
            print(f"Filled missing values in '{col}' with mean: {mean_val:.2f}")

    # B. Mode Fill for Categorical Columns (Mean doesn't work for categories like 'thal')
    # We must fill categorical NaNs before One-Hot Encoding
    for col in cat_cols:
        print(f"Processing categorical column: '{col}'")
        if df[col].isnull().sum() > 0:
            df[col] = df[col].astype('category')
            mode_val = df[col].mode()[0]
            df[col] = df[col].fillna(mode_val)
            print(f"Filled missing values in '{col}' with mode: {mode_val}")

    print (df.info())
    print(df.head())
    
    df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)
    
    # 6. Feature Reduction (Correlation Analysis)
    print("Analyzing feature importance...")

    print("\nStarting Feature Analysis...")
    # Calculate correlation of all features with the 'target'
    correlations = df_encoded.corr()['target'].abs().sort_values(ascending=False)
    
    print("\nFeature Correlations with Target (sorted by importance):")
    print(correlations)

    threshold = 0.1 

    selected_features = correlations[correlations > threshold].index.tolist()
    dropped_features = correlations[correlations <= threshold].index.tolist()

    print(f"\n[Analysis Result] Keeping features with correlation > {threshold}")
    print(f"Dropped Features: {dropped_features}")
    print(f"Selected Features: {selected_features}")

    df_final = df_encoded[selected_features]

    X_train, X_test, y_train, y_test = train_test_split(df_final.drop('target', axis=1), df_final['target'], test_size=0.1, random_state=42, stratify = df_final['target'])

    X_train['target'] = y_train
    X_test['target'] = y_test

    X_train.to_csv('dataset\\train.csv', index=False)
    X_test.to_csv('dataset\\test.csv', index=False)


In [3]:
# this dataset has to be downloaded as it is included in the .gitignore file
prepare_dataset('dataset\\main\\heart_disease_uci.csv')

Loading dataset...
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 920 entries, 0 to 919
Data columns (total 16 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   id        920 non-null    int64  
 1   age       920 non-null    int64  
 2   sex       920 non-null    object 
 3   dataset   920 non-null    object 
 4   cp        920 non-null    object 
 5   trestbps  861 non-null    float64
 6   chol      890 non-null    float64
 7   fbs       830 non-null    object 
 8   restecg   918 non-null    object 
 9   thalch    865 non-null    float64
 10  exang     865 non-null    object 
 11  oldpeak   858 non-null    float64
 12  slope     611 non-null    object 
 13  ca        309 non-null    float64
 14  thal      434 non-null    object 
 15  num       920 non-null    int64  
dtypes: float64(5), int64(3), object(8)
memory usage: 115.1+ KB
None
     age  trestbps   chol  thalch  oldpeak   ca  target
0     63     145.0  233.0   150.0      2.3  0.0  